In [1]:
import pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark_job = SparkSession.builder.appName("Data Analysis Job").getOrCreate()
spark_job

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### DATA INSPECTION

In [4]:
raw_df = spark_job.read.option('header', 'true').csv('salaries.csv', inferSchema=True)
raw_df.show()

+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+----------+-------------+--------------------+-----------------+
|   entidadfederativa|      sujetoobligado|              nombre|        denominacion|montoneto|               cargo|                area|montobruto|idInformacion|periodoreportainicio|periodoreportafin|
+--------------------+--------------------+--------------------+--------------------+---------+--------------------+--------------------+----------+-------------+--------------------+-----------------+
|             Hidalgo|            Jaltocán|Adolfo Hernandez ...|           Fontanero|   4000.0|           Fontanero|      OBRAS PUBLICAS|    4254.0|     16311845|          01/01/2018|       30/06/2018|
|    Ciudad de México| Secretaría de Salud|ARELY SAMANTA CLE...|"AUXILIAR DE ENFE...| 12177.86|"AUXILIAR DE ENFE...|H.G. ENRIQUE CABRERA|   16092.0|     16480190|          01/01/2018|       31

### DATA CLEANING 

In [7]:
# Null Count
null_count = raw_df.count() - raw_df.na.drop().count()
null_count

187975

In [8]:
# checking for null percent compared to original dataset count
null_count_pct = (null_count/raw_df.count())*100
null_count_pct

9.350682814989545

### USING THE SALARIES COLUMN MONTOBRUTO AND MONTONETO TO ACCOUNT FOR NULL VALUES 

In [9]:
raw_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: string (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: string (nullable = true)
 |-- periodoreportafin: string (nullable = true)



In [ ]:
# only care about rows having actual salary figures which is the core of this dataset
raw_df.na.drop(subset=['montobruto', 'montoneto']).count()

1861815

In [12]:
pct_with_salaries = (raw_df.na.drop(subset=['montobruto', 'montoneto']).count())/(raw_df.count())*100
pct_with_salaries

92.61466431807295

Showing top states with largest null count

In [13]:
dropped_rows = raw_df.subtract(raw_df.na.drop(subset=['montobruto', 'montoneto']))
dropped_rows.groupBy('entidadfederativa').count().orderBy('count', ascending=False).show()

+--------------------+-----+
|   entidadfederativa|count|
+--------------------+-----+
|             Jalisco|27641|
|     Baja California|25384|
|            Guerrero|15813|
|     San Luis Potosí| 6006|
|              Sonora| 4383|
|          Federación| 3071|
|      Aguascalientes| 2316|
|              Colima| 1070|
|             Hidalgo|  934|
|             Chiapas|  574|
|Coahuila de Zaragoza|  539|
|    Ciudad de México|  223|
|           Chihuahua|  177|
| Michoacán de Ocampo|  124|
|          Guanajuato|  115|
|             Morelos|   84|
|             Tabasco|   49|
| Baja California Sur|   37|
|          Nuevo León|   36|
|           Querétaro|   24|
+--------------------+-----+
only showing top 20 rows


### DATA VALIDATION

In [14]:
raw_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: string (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: string (nullable = true)
 |-- periodoreportafin: string (nullable = true)



We need to convert to montobruto and montoneto to double and both periods to datetime

In [15]:
from pyspark.sql.functions import to_date

raw_df = raw_df.withColumn(
        "periodoreportainicio",
        to_date("periodoreportainicio", "dd/MM/yyyy"))

In [16]:
spark_df = raw_df.withColumn('periodoreportafin', 
                             to_date('periodoreportafin', 'dd/MM/yyyy'))

In [17]:
from pyspark.sql.functions import col

spark_df = (
    spark_df
    .withColumn("montoneto", col("montoneto").cast("double"))
    .withColumn("montobruto", col("montobruto").cast("double"))
)

In [20]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)



Ran into series of problems because of format inconsistency in the earlier notebooks so I included a preprocessing step for validating the montoneto and montoburo columns

In [30]:
from pyspark.sql import functions as F

# 1. Read raw CSV with inferSchema disabled
raw_df = (
    spark_job.read.option("header", "true")
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .csv(r"C:\Users\user\Desktop\Data Engineering\Pyspark Projects\Salaries Dataset\salaries.csv")
)

# 2. Clean salaries and sanitize dates as strings (avoiding strict to_date parsers entirely)
clean_df = raw_df.withColumn(
    "montobruto",
    F.when(
        F.trim(F.col("montobruto")).rlike(r"^[0-9,.\s-]+$"),
        F.regexp_replace(F.trim(F.col("montobruto")), r",", "").cast("double")
    ).otherwise(None)
).withColumn(
    "montoneto",
    F.when(
        F.trim(F.col("montoneto")).rlike(r"^[0-9,.\s-]+$"),
        F.regexp_replace(F.trim(F.col("montoneto")), r",", "").cast("double")
    ).otherwise(None)
).withColumn(
    "periodoreportainicio",
    F.when(
        F.trim(F.col("periodoreportainicio")).rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$"),
        F.trim(F.col("periodoreportainicio"))
    ).otherwise(None)
).withColumn(
    "periodoreportafin",
    F.when(
        F.trim(F.col("periodoreportafin")).rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$"),
        F.trim(F.col("periodoreportafin"))
    ).otherwise(None)
)

# 3. Drop rows where key fields are missing/null
clean_df = clean_df.filter(
    F.col("montobruto").isNotNull()
    & F.col("montoneto").isNotNull()
    & F.col("periodoreportainicio").isNotNull()
    & F.col("periodoreportafin").isNotNull()
)

clean_df.select(
    "montobruto", "montoneto", "periodoreportainicio", "periodoreportafin"
).show(20, truncate=False)

+----------+---------+--------------------+-----------------+
|montobruto|montoneto|periodoreportainicio|periodoreportafin|
+----------+---------+--------------------+-----------------+
|4254.0    |4000.0   |01/01/2018          |30/06/2018       |
|16092.0   |12177.86 |01/01/2018          |31/03/2018       |
|16030.0   |11652.0  |01/01/2020          |31/03/2020       |
|2910.65   |10180.57 |01/07/2018          |31/12/2018       |
|6188.4    |17004.4  |01/07/2019          |31/12/2019       |
|2982.2    |2644.0   |01/07/2018          |31/12/2018       |
|4898.28   |4898.28  |01/07/2019          |31/12/2019       |
|619.0     |474.96   |01/05/2019          |31/05/2019       |
|16812.67  |13109.02 |01/04/2018          |30/06/2018       |
|7579.22   |6299.3   |01/04/2018          |30/06/2018       |
|3974.8    |13730.55 |01/07/2018          |31/12/2018       |
|13280.15  |11753.87 |01/04/2019          |30/06/2019       |
|13198.8   |12000.0  |01/08/2019          |31/08/2019       |
|3918.15

In [26]:
from pyspark.sql import functions as F

# Read raw CSV as text first
raw_df = spark_job.read.option("header", "true").option("inferSchema", "false").csv(
    r"C:\Users\user\Desktop\Data Engineering\Pyspark Projects\Salaries Dataset\salaries.csv"
)

raw_df.show(10, truncate=False)
raw_df.printSchema()

# Check whether commas or semicolons are the real delimiter
# If columns look shifted, try:
# raw_df = spark.read.option("header", "true").option("sep", ";").csv(...)
# raw_df.show(10, truncate=False)

# Find bad rows before casting
bad_salary_rows = raw_df.filter(
    F.trim(F.col("montobruto").cast("string")).rlike(r".*[A-Za-z].*")
    |
    F.trim(F.col("montoneto").cast("string")).rlike(r".*[A-Za-z].*")
    |
    F.trim(F.col("montobruto").cast("string")).isin("USET", "NULL", "NA", "")
    |
    F.trim(F.col("montoneto").cast("string")).isin("USET", "NULL", "NA", "")
    |
    F.trim(F.col("montobruto").cast("string")).rlike(r".*\d{2}/\d{2}/\d{4}.*")
    |
    F.trim(F.col("montoneto").cast("string")).rlike(r".*\d{2}/\d{2}/\d{4}.*")
)

bad_salary_rows.select(
    "entidadfederativa",
    "montobruto",
    "montoneto",
    "periodoreportainicio",
    "periodoreportafin"
).show(50, truncate=False)

# Safe conversion: only numeric-like strings become doubles, everything else becomes null
clean_df = raw_df.withColumn(
    "montobruto",
    F.when(
        F.trim(F.col("montobruto").cast("string")).rlike(r"^[0-9,.\s-]+$"),
        F.regexp_replace(F.trim(F.col("montobruto").cast("string")), r",", "").cast("double")
    ).otherwise(None)
).withColumn(
    "montoneto",
    F.when(
        F.trim(F.col("montoneto").cast("string")).rlike(r"^[0-9,.\s-]+$"),
        F.regexp_replace(F.trim(F.col("montoneto").cast("string")), r",", "").cast("double")
    ).otherwise(None)
)

# Remove rows with missing salary values
clean_df = clean_df.filter(
    F.col("montobruto").isNotNull() &
    F.col("montoneto").isNotNull()
)

clean_df.select("montobruto", "montoneto").show(20, truncate=False)

+-----------------+----------------------------------------------------------+------------------------------+------------------------------------------------------------+---------+------------------------------------------------------------+-------------------------------------------------------------------+----------+-------------+--------------------+-----------------+
|entidadfederativa|sujetoobligado                                            |nombre                        |denominacion                                                |montoneto|cargo                                                       |area                                                               |montobruto|idInformacion|periodoreportainicio|periodoreportafin|
+-----------------+----------------------------------------------------------+------------------------------+------------------------------------------------------------+---------+------------------------------------------------------------+-----------

### FEATURE ENGINEERING
1. Calculating for time difference from report start date (periodoreportainicio) and end of report end date (periodoreportafin)
2. Calculating Money deducted by subtracting Net Salary (montoneto) from Gross Salary (montobruto)

In [31]:
from pyspark.sql import functions as F

# convert valid date strings to real dates
clean_df = clean_df.withColumn(
    "periodoreportainicio",
    F.when(
        F.col("periodoreportainicio").rlike(r"^\d{2}/\d{2}/\d{4}$"),
        F.to_date("periodoreportainicio", "dd/MM/yyyy")
    ).otherwise(None)
).withColumn(
    "periodoreportafin",
    F.when(
        F.col("periodoreportafin").rlike(r"^\d{2}/\d{2}/\d{4}$"),
        F.to_date("periodoreportafin", "dd/MM/yyyy")
    ).otherwise(None)
)

# keep only rows with clean salary + valid dates
clean_df = clean_df.filter(
    F.col("montobruto").isNotNull() &
    F.col("montoneto").isNotNull() &
    F.col("periodoreportainicio").isNotNull() &
    F.col("periodoreportafin").isNotNull()
)

# feature engineering
clean_df = clean_df.withColumn(
    "period_dias",
    F.datediff("periodoreportafin", "periodoreportainicio")
).withColumn(
    "deducciones_estimadas",
    F.col("montobruto") - F.col("montoneto")
)

# validate
clean_df.printSchema()
clean_df.select(
    "montobruto", "montoneto",
    "periodoreportainicio", "periodoreportafin",
    "period_dias", "deducciones_estimadas"
).show(20, truncate=False)

# EDA
clean_df.select("montobruto", "montoneto", "period_dias", "deducciones_estimadas").describe().show()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)
 |-- deducciones_estimadas: double (nullable = true)

+----------+---------+--------------------+-----------------+-----------+---------------------+
|montobruto|montoneto|periodoreportainicio|periodoreportafin|period_dias|deducciones_estimadas|
+----------+---------+--------------------+-----------------+-----------+---------------------+
|4254.0    |4000.0   |2018-01-01          |2018-06-30       |180        |254.0                |
|16092.0   |12177.86 |2018-01-01      

### EXPLORATORY DATA ANALYSIS

In [21]:
from pyspark.sql import functions as F

spark_df.printSchema()
spark_df.columns
spark_df.select("montobruto", "montoneto").show(20, truncate=False)

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)
 |-- deducciones_estimadas: double (nullable = true)

+----------+---------+
|montobruto|montoneto|
+----------+---------+
|4254.0    |4000.0   |
|16092.0   |12177.86 |
|16030.0   |11652.0  |
|2910.65   |10180.57 |
|6188.4    |17004.4  |
|2982.2    |2644.0   |
|4898.28   |4898.28  |
|619.0     |474.96   |
|16812.67  |13109.02 |
|7579.22   |6299.3   |
|3974.8    |13730.55 |
|13280.15  |11753.87 |
|13198.8   |12000.0  |
|3918.15   |3673.91  |
|14861.81  |12050.8  |
|27920.1

In [22]:
from pyspark.sql import functions as F

spark_df.select("montobruto").show(20, truncate=False)
spark_df.filter(F.col("montobruto").isNull()).count()
spark_df.filter(F.trim(F.col("montobruto").cast("string")) == "").count()
spark_df.printSchema()

+----------+
|montobruto|
+----------+
|4254.0    |
|16092.0   |
|16030.0   |
|2910.65   |
|6188.4    |
|2982.2    |
|4898.28   |
|619.0     |
|16812.67  |
|7579.22   |
|3974.8    |
|13280.15  |
|13198.8   |
|3918.15   |
|14861.81  |
|27920.1   |
|40924.26  |
|11146.0   |
|14332.31  |
|12966.37  |
+----------+
only showing top 20 rows


{"ts": "2026-09-10 09:05:06.606", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '31/03/2019' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [15]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o152.count.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '31/03/2019' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 6 in cell [15]\n\r\n\tat org.apache.spark.sql.errors.Query

NumberFormatException: [CAST_INVALID_INPUT] The value '31/03/2019' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 6 in cell [15]


In [19]:
spark_df.select('montobruto').describe().show()

quantiles = spark_df.approxQuantile('montobruto', [0.25, 0.5, 0.75, 0.9], 0.01)
print(quantiles)

{"ts": "2026-09-10 08:26:20.749", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' H. MATAMOROS TAM.\"' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 6 in cell [15]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o105.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' H. MATAMOROS TAM.\"' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 6 in cell [15]\n\r\n\tat org.apac

NumberFormatException: [CAST_INVALID_INPUT] The value ' H. MATAMOROS TAM."' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 6 in cell [15]
